# Análise SQL — Reclamações Localiza no Reclame Aqui

## Seção 1 — Setup

Os dados são carregados em uma base **SQLite em memória**, permitindo análise via SQL puro, sem pandas para as consultas.  
Cada seção responde a uma pergunta de negócio diretamente com uma query.

**Arquivo de origem:** `dados/reclamacoes_processadas.csv`

In [ ]:
import sqlite3
import pandas as pd

df = pd.read_csv('dados/reclamacoes_processadas.csv')
conn = sqlite3.connect(':memory:')
df.to_sql('reclamacoes', conn, index=False)
print(f"Base carregada: {len(df)} reclamações")

## Seção 2 — Visão geral por status

Qual a distribuição das reclamações por status e qual o peso de cada um no total?

In [ ]:
query = """
SELECT
    status,
    COUNT(*) AS total,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM reclamacoes), 1) AS percentual
FROM reclamacoes
GROUP BY status
ORDER BY total DESC
"""
display(pd.read_sql(query, conn))

A maioria das reclamações está concentrada nos status **Não Respondida** e **Resolvido**, indicando que a Localiza responde boa parte dos casos, mas ainda deixa um volume relevante sem retorno. O percentual de casos **Não Resolvidos** representa um risco direto à percepção de marca, pois são clientes que receberam resposta mas não tiveram seu problema solucionado.

## Seção 3 — Taxa de resolução por categoria

Quais categorias têm as maiores oportunidades de melhoria considerando volume e taxa de resolução?

In [ ]:
query = """
SELECT
    categoria,
    COUNT(*) AS total,
    SUM(CASE WHEN status = 'RESOLVIDO' THEN 1 ELSE 0 END) AS resolvidos,
    ROUND(100.0 * SUM(CASE WHEN status = 'RESOLVIDO' THEN 1 ELSE 0 END) / COUNT(*), 1) AS taxa_resolucao,
    ROUND(AVG(nota_cliente), 2) AS nota_media
FROM reclamacoes
GROUP BY categoria
ORDER BY taxa_resolucao ASC
"""
display(pd.read_sql(query, conn))

A categoria **Problema Financeiro** combina o maior volume absoluto com uma taxa de resolução relativamente baixa, tornando-a a principal oportunidade de melhoria operacional. Categorias como **Bloqueio de Cadastro** e **Falha no App / Reserva** também merecem atenção por reunirem alto volume e notas médias baixas, sugerindo que o cliente não percebe progresso mesmo quando há resposta.

## Seção 4 — Estados com nota abaixo da média geral

Quais estados performam abaixo da média nacional de satisfação?

In [ ]:
query = """
SELECT
    estado,
    COUNT(*) AS total_reclamacoes,
    ROUND(AVG(nota_cliente), 2) AS nota_media
FROM reclamacoes
WHERE nota_cliente IS NOT NULL
GROUP BY estado
HAVING AVG(nota_cliente) < (SELECT AVG(nota_cliente) FROM reclamacoes WHERE nota_cliente IS NOT NULL)
ORDER BY nota_media ASC
"""
display(pd.read_sql(query, conn))

Os estados com nota abaixo da média nacional merecem atenção operacional prioritária. Estados com alto volume de reclamações e nota baixa simultaneamente, como **SP** e **MG**,indicam problemas sistêmicos que vão além de casos isolados, sugerindo falhas em processos ou unidades específicas nessas regiões.

## Seção 5 — Comparativo Belo Horizonte vs restante de Minas Gerais

O desempenho de Belo Horizonte puxa a média do estado para baixo ou segue o mesmo padrão?

In [ ]:
query = """
SELECT
    CASE WHEN cidade = 'Belo Horizonte' THEN 'Belo Horizonte' ELSE 'Restante de MG' END AS recorte,
    COUNT(*) AS total,
    ROUND(AVG(nota_cliente), 2) AS nota_media,
    ROUND(100.0 * SUM(CASE WHEN status = 'RESOLVIDO' THEN 1 ELSE 0 END) / COUNT(*), 1) AS taxa_resolucao,
    SUM(CASE WHEN status = 'NÃO RESPONDIDA' THEN 1 ELSE 0 END) AS nao_respondidas
FROM reclamacoes
WHERE estado = 'MG'
GROUP BY recorte
"""
display(pd.read_sql(query, conn))

A comparação revela se Belo Horizonte apresenta um padrão de atendimento distinto do restante do estado. Caso a taxa de resolução de BH seja inferior e a nota média mais baixa, isso indica que as unidades da capital enfrentam desafios operacionais próprios, possivelmente relacionados a maior volume e menor capacidade de atendimento individualizado.

## Seção 6 — Categorias críticas: alto volume e baixa resolução

Combinando volume e taxa de resolução, onde está o maior risco operacional?

In [ ]:
query = """
SELECT
    categoria,
    COUNT(*) AS total,
    ROUND(100.0 * SUM(CASE WHEN status = 'RESOLVIDO' THEN 1 ELSE 0 END) / COUNT(*), 1) AS taxa_resolucao,
    ROUND(AVG(nota_cliente), 2) AS nota_media,
    SUM(CASE WHEN status = 'NÃO RESPONDIDA' THEN 1 ELSE 0 END) AS sem_resposta
FROM reclamacoes
GROUP BY categoria
HAVING COUNT(*) > 10
ORDER BY taxa_resolucao ASC, total DESC
"""
display(pd.read_sql(query, conn))

O `HAVING COUNT(*) > 10` foi usado para filtrar categorias com volume relevante, evitando que categorias com poucos registros distorçam a análise de risco. Categorias com baixo volume podem ter taxas de resolução extremas (0% ou 100%) por simples acaso estatístico, o filtro garante que apenas padrões consistentes sejam considerados na tomada de decisão.

## Seção 7 — Cidades com maior concentração de reclamações sem resposta

Quais cidades concentram mais casos ignorados?

In [ ]:
query = """
SELECT
    cidade,
    estado,
    COUNT(*) AS total_nao_respondidas,
    ROUND(100.0 * COUNT(*) / (
        SELECT COUNT(*) FROM reclamacoes WHERE cidade = r.cidade
    ), 1) AS percentual_da_cidade
FROM reclamacoes r
WHERE status = 'NÃO RESPONDIDA'
GROUP BY cidade, estado
HAVING COUNT(*) >= 3
ORDER BY total_nao_respondidas DESC
LIMIT 10
"""
display(pd.read_sql(query, conn))

O padrão geográfico das reclamações sem resposta tende a seguir o volume geral, grandes centros como São Paulo e Rio de Janeiro dominam em números absolutos. Contudo, o **percentual sobre o total da cidade** é o indicador mais revelador: cidades com alto percentual de casos ignorados indicam possível sobrecarga ou negligência de unidades locais, independentemente do tamanho do mercado.

## Seção 8 — Efeito da resolução na nota do cliente

A resolução do problema é o principal fator de satisfação ou outros elementos influenciam?

In [ ]:
query = """
SELECT
    status,
    COUNT(*) AS total_com_nota,
    ROUND(AVG(nota_cliente), 2) AS nota_media,
    ROUND(MIN(nota_cliente), 1) AS nota_minima,
    ROUND(MAX(nota_cliente), 1) AS nota_maxima
FROM reclamacoes
WHERE nota_cliente IS NOT NULL
GROUP BY status
ORDER BY nota_media DESC
"""
display(pd.read_sql(query, conn))

A resolução do problema é claramente o principal driver de satisfação: casos com status **Resolvido** apresentam nota média significativamente superior aos **Não Resolvidos**. A diferença entre os extremos reforça que o cliente não avalia apenas a velocidade do atendimento, mas sim o desfecho, resolver o problema é inegociável para converter uma experiência negativa em percepção positiva de marca.

In [ ]:
conn.close()
print("Conexão encerrada.")